# 04b - Regularisasi: Ridge, LASSO, dan Jaring Elastis (Elastic Net)
Materi Modul 4: regresi Ridge, regresi Lasso (Bagian 1 & 2), jaring elastis.

Regularisasi = menambahkan **penalti** pada ukuran koefisien agar model tidak terlalu kompleks.

| Model | Penalti | Efek |
|---|---|---|
| Ridge (L2) | $\alpha \sum \beta_j^2$ | Mengecilkan koefisien, tidak sampai 0 |
| Lasso (L1) | $\alpha \sum |\beta_j|$ | Bisa membuat koefisien tepat 0 (seleksi fitur) |
| Elastic Net | campuran L1 dan L2 | Kompromi keduanya |

**Penting:** fitur harus di-scale dulu (pakai `StandardScaler`) karena penalti sensitif terhadap skala.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, RidgeCV, LassoCV, ElasticNetCV
from sklearn.metrics import mean_squared_error

df = pd.read_csv('../datasets/diabetes.csv')
X = df.drop(columns='progres_penyakit')
y = df['progres_penyakit']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

def rmse(model):
    return np.sqrt(mean_squared_error(y_test, model.predict(X_test)))

## 1. Bandingkan 4 model

In [ ]:
models = {
    'Linear (tanpa regularisasi)': make_pipeline(StandardScaler(), LinearRegression()),
    'Ridge (alpha=10)': make_pipeline(StandardScaler(), Ridge(alpha=10)),
    'Lasso (alpha=1)': make_pipeline(StandardScaler(), Lasso(alpha=1)),
    'ElasticNet (alpha=1, l1_ratio=0.5)': make_pipeline(StandardScaler(), ElasticNet(alpha=1, l1_ratio=0.5)),
}

for nama, m in models.items():
    m.fit(X_train, y_train)
    print(f'{nama:38s} | RMSE test = {rmse(m):.2f}')

## 2. Koefisien tiap model

In [ ]:
koef = pd.DataFrame({nama: m[-1].coef_ for nama, m in models.items()}, index=X.columns).round(2)
koef

In [ ]:
koef.plot(kind='bar', figsize=(12, 4))
plt.axhline(0, color='black', lw=0.8)
plt.title('Koefisien: Linear vs Ridge vs Lasso vs ElasticNet')
plt.ylabel('Koefisien (data ter-scale)')
plt.legend(fontsize=8)
plt.show()

## 3. Pengaruh nilai alpha pada koefisien (coefficient path)

In [ ]:
alphas = np.logspace(-2, 3, 50)
Xs_train = StandardScaler().fit_transform(X_train)

koef_ridge, koef_lasso = [], []
for a in alphas:
    koef_ridge.append(Ridge(alpha=a).fit(Xs_train, y_train).coef_)
    koef_lasso.append(Lasso(alpha=a / 10, max_iter=10000).fit(Xs_train, y_train).coef_)

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].plot(alphas, koef_ridge); ax[0].set_xscale('log'); ax[0].set_title('Ridge: koefisien mengecil pelan-pelan')
ax[1].plot(alphas / 10, koef_lasso); ax[1].set_xscale('log'); ax[1].set_title('Lasso: koefisien jadi 0 satu per satu')
for a in ax:
    a.set_xlabel('alpha'); a.set_ylabel('Koefisien')
ax[1].legend(X.columns, fontsize=7, ncol=2)
plt.tight_layout(); plt.show()

## 4. Pengaruh alpha pada RMSE test

In [ ]:
rmse_ridge, rmse_lasso = [], []
for a in alphas:
    rmse_ridge.append(rmse(make_pipeline(StandardScaler(), Ridge(alpha=a)).fit(X_train, y_train)))
    rmse_lasso.append(rmse(make_pipeline(StandardScaler(), Lasso(alpha=a / 10, max_iter=10000)).fit(X_train, y_train)))

plt.plot(alphas, rmse_ridge, label='Ridge')
plt.plot(alphas, rmse_lasso, label='Lasso (alpha/10)')
plt.xscale('log'); plt.xlabel('alpha'); plt.ylabel('RMSE test'); plt.legend()
plt.title('Alpha terlalu kecil = overfit, terlalu besar = underfit')
plt.show()

## 5. Seleksi fitur dengan Lasso

In [ ]:
lasso = make_pipeline(StandardScaler(), Lasso(alpha=2)).fit(X_train, y_train)
koef_lasso_final = pd.Series(lasso[-1].coef_, index=X.columns)
print('Fitur yang dibuang (koefisien = 0):', list(koef_lasso_final[koef_lasso_final == 0].index))
print('Fitur yang dipertahankan          :', list(koef_lasso_final[koef_lasso_final != 0].index))

## 6. Memilih alpha otomatis dengan validasi silang (`RidgeCV`, `LassoCV`, `ElasticNetCV`)

In [ ]:
ridge_cv = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 3, 50), cv=5)).fit(X_train, y_train)
lasso_cv = make_pipeline(StandardScaler(), LassoCV(cv=5, random_state=0, max_iter=10000)).fit(X_train, y_train)
enet_cv = make_pipeline(StandardScaler(), ElasticNetCV(l1_ratio=[0.1, 0.5, 0.7, 0.9, 1], cv=5, random_state=0, max_iter=10000)).fit(X_train, y_train)

print('RidgeCV      | alpha terbaik =', round(ridge_cv[-1].alpha_, 3), '| RMSE test =', round(rmse(ridge_cv), 2))
print('LassoCV      | alpha terbaik =', round(lasso_cv[-1].alpha_, 3), '| RMSE test =', round(rmse(lasso_cv), 2))
print('ElasticNetCV | alpha =', round(enet_cv[-1].alpha_, 3), '| l1_ratio =', enet_cv[-1].l1_ratio_, '| RMSE test =', round(rmse(enet_cv), 2))

## Catatan
- (kapan pakai Ridge vs Lasso vs Elastic Net, kenapa harus di-scale, dll.)